# CSCIE89 HW05 – Problem 3: Transfer learning with VGG16 on the flower dataset

This notebook starts from **3_FineTune_VGG16_PyTorch_Cat_Dog** and keeps only the sections needed for Problem 3, with the same code and structure. The changes are:

- **Flower dataset instead of cats and dogs:** the Google Drive + local Colab setup from Problem 2, and the same stratified 70% / 15% / 15% split (2,569 / 550 / 551 images).
- **Five classes instead of two:** the final layer has 5 outputs, `BCEWithLogitsLoss` becomes `CrossEntropyLoss`, and predictions use `argmax` instead of `logits >= 0`.
- **Precision of the stage 1 and stage 2 models on the entire test set** (section 10).
- **One test image of every flower class** (section 11).

Steps (same as the class notebook):

1. Load and prepare the images.
2. Load pretrained VGG16 and add a new classifier.
3. Stage 1: freeze the convolutional layers and train only the classifier.
4. Stage 2: unfreeze the final convolutional block and fine-tune it with a smaller learning rate.
5. Evaluate both stages on the unused test images.

The first run downloads the flower dataset and roughly 528 MB of VGG16 weights. Use a GPU in Colab (**Runtime → Change runtime type → GPU**); training on a CPU is slow.

## 1. Import tools and choose training settings

PyTorch handles training; torchvision provides VGG16 and image tools. Scikit-learn helps split the data and report results.

- An **epoch** is one pass through all training images.
- A **batch** is a small group of images processed together.
- The **learning rate** controls the size of weight updates.
- The **seed** makes the data split repeatable; exact training results can still vary.
- The code selects an NVIDIA GPU, an Apple GPU, or a CPU, in that order.

`precision_score` is added to compute overall precision for Problem 3.

In [ ]:
# Run once if packages are missing, then restart the kernel:
# %pip install torch torchvision scikit-learn matplotlib pillow

from pathlib import Path                         # Tool for building folder/file paths.
import copy                                      # Lets us save an independent copy of the model's weights.
import torch                                     # PyTorch: the deep-learning library.
from torch import nn                             # Building blocks for networks (layers, loss functions).
from torch.utils.data import DataLoader, Subset  # DataLoader feeds images in batches; Subset picks some images by their index.
from torchvision import datasets, models, transforms  # Image-folder reader, pretrained models (VGG16), image resizing/augmenting.
from torchvision.datasets.utils import download_and_extract_archive  # Downloads a zip/tgz file and unpacks it.
from sklearn.model_selection import train_test_split  # Splits data into train/validation/test.
from sklearn.metrics import classification_report, precision_score  # Precision/recall table, and overall precision.
import matplotlib.pyplot as plt                  # For showing images.
from PIL import Image                            # For opening image files.

SEED = 42                                        # Fixed random seed; same as Problem 2, so we get the same data split.
IMAGE_SIZE = 150                                 # Every photo is resized to 150 x 150 pixels.
BATCH_SIZE = 20                                  # The model looks at 20 photos at a time.
INITIAL_EPOCHS = 5                               # Stage 1: up to 5 passes over the training photos.
FINETUNE_EPOCHS = 3                              # Stage 2: up to 3 more passes.
INITIAL_LR = 2e-5                                # Stage 1 learning rate (size of each weight update).
FINETUNE_LR = 1e-5                               # Stage 2 learning rate: smaller, so VGG16's weights change gently.
PATIENCE = 3                                     # Stop a stage early if validation loss doesn't improve for 3 epochs.

torch.manual_seed(SEED)                          # Make PyTorch's random choices repeatable.
torch.set_num_threads(4)                         # Limit CPU threads (only matters when there is no GPU).
device = torch.device('cuda' if torch.cuda.is_available() else          # Use the NVIDIA GPU if there is one (Colab),
                      'mps' if torch.backends.mps.is_available() else 'cpu')  # else an Apple GPU, else the CPU.
print('Using device:', device)                   # Show which one we are using.

## 2. Google Drive + local Colab setup: load the flower images

This is the same setup as Problem 2:

- A **permanent copy** of the photos is kept in Google Drive (`DRIVE_DATA_DIR`).
- Training reads the photos from Colab's **fast local disk** (`DATA_DIR = /content/data`).
- The first time, the dataset is downloaded into Colab and copied to Drive. Later sessions copy it from Drive instead of downloading it again.

The [Flower Photos dataset](https://www.tensorflow.org/tutorials/images/classification) has 3,670 photographs in five folders: daisy, dandelion, roses, sunflowers, and tulips.

In [ ]:
# GOOGLE DRIVE + LOCAL COLAB SETUP

from google.colab import drive                   # Colab tool to connect your Google Drive.
import shutil                                    # For copying whole folders.

drive.mount('/content/drive')                    # Connect Google Drive (asks for permission the first time).

# Permanent location in Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW05')  # Your homework folder in Drive.
DRIVE_DATA_DIR = PROJECT_DIR / 'data'                                       # Data folder in Drive (kept forever).
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)                           # Create it if it doesn't exist yet.

# IMPORTANT: # DATA_DIR is the fast temporary Colab drive

DATA_DIR = Path('/content/data')                                            # Colab's own disk: fast, but erased when the session ends.
DATA_DIR.mkdir(parents=True, exist_ok=True)                                 # Create it if it doesn't exist yet.


flower_root = DATA_DIR / 'flower_photos'                                    # Where training reads the flowers from (fast disk).
drive_flower_root = DRIVE_DATA_DIR / 'flower_photos'                        # Where the permanent copy lives (Drive).

# If flowers are not already in Colab
if not flower_root.exists():

    # Copy from Google Drive if already saved there
    if drive_flower_root.exists():
        shutil.copytree(drive_flower_root, flower_root)  # Copy Drive -> Colab disk (faster than downloading).

    # Otherwise download them into Colab
    else:
        download_and_extract_archive(            # Download the flower archive and unpack it:
            'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',  # from this web address,
            download_root=str(DATA_DIR),         # into the Colab disk.
        )

# Save a permanent copy to Google Drive if one does not exist
if not drive_flower_root.exists():
    shutil.copytree(flower_root, drive_flower_root)  # Copy Colab disk -> Drive (happens only the first time).

print('Flower directory:', flower_root)          # Show where the photos are.

## 3. Prepare images for VGG16

Every image is resized to 150 × 150 pixels so images can be grouped into batches. `ToTensor()` converts an RGB image to three numeric channels with values from 0 to 1. `Normalize()` then adjusts each channel using the ImageNet mean and standard deviation expected by the pretrained weights.

Training images also receive random flips and small rotations. This data augmentation gives the model varied examples. Validation and test images are processed consistently, without random changes.

`ImageFolder` reads images from class-named folders and assigns their numeric labels. The flowers have a single folder, so we create **two views** of it: one with the training transform and one with the evaluation transform. Section 4 decides which images each view supplies.

In [ ]:
MEAN = [0.485, 0.456, 0.406]                     # Average red, green, blue values of ImageNet photos.
STD = [0.229, 0.224, 0.225]                      # Spread of red, green, blue values of ImageNet photos.

train_transform = transforms.Compose([           # Steps applied to each TRAINING photo, in order:
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # 1. Resize to 150 x 150.
    transforms.RandomHorizontalFlip(),     # 2. Mirror left-right, at random (half the time).
    transforms.RandomRotation(10),         # 3. Turn by a random angle between -10 and +10 degrees.
    transforms.ToTensor(),                # 4. Turn the picture into numbers between 0 and 1.
    transforms.Normalize(MEAN, STD),       # 5. Rescale those numbers the same way VGG16 saw ImageNet photos.
])

eval_transform = transforms.Compose([            # Steps applied to VALIDATION and TEST photos (no random changes):
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # 1. Resize to 150 x 150.
    transforms.ToTensor(),                       # 2. Turn the picture into numbers between 0 and 1.
    transforms.Normalize(MEAN, STD),             # 3. Rescale like ImageNet.
])

# ImageFolder assigns integer labels using alphabetically sorted folder names.
train_view = datasets.ImageFolder(flower_root, transform=train_transform)  # All 3,670 flowers, read with the TRAINING steps.
eval_view = datasets.ImageFolder(flower_root, transform=eval_transform)    # The same 3,670 flowers, read with the TEST steps.
class_names = train_view.classes                 # The 5 class names (folder names), in label order.
num_classes = len(class_names)                   # 5.
assert train_view.class_to_idx == eval_view.class_to_idx  # Safety check: both views give each class the same number.
print('Class mapping:', train_view.class_to_idx) # daisy=0, dandelion=1, roses=2, sunflowers=3, tulips=4

## 4. Separate training, validation, and test data

We use the same stratified split as Problem 2 (same seed and proportions):

- **Training:** 2,569 images (70%) used to update weights.
- **Validation:** 550 images (15%) used to select the best weights and decide when to stop.
- **Test:** 551 images (15%) reserved for the final evaluation, not for choosing settings.

The stratified split keeps the class proportions balanced. `Subset` selects images by index. `DataLoader` supplies batches; only the training batches are shuffled.

In [ ]:
all_indices = list(range(len(eval_view)))        # Number every photo: 0, 1, 2, ..., 3669.
train_indices, remaining = train_test_split(     # Split 1: put 70% of the photo numbers in training, keep 30% aside.
    all_indices, test_size=0.30,                  # All photo numbers; 30% go to the 'aside' part.
    random_state=SEED, stratify=eval_view.targets,  # Same split every run; each part keeps the same mix of flower types.
)
val_indices, test_indices = train_test_split(    # Split 2: cut the 30% in half -> 15% validation, 15% test.
    remaining, test_size=0.50,                    # The 30% set aside; half of it goes to test.
    random_state=SEED, stratify=[eval_view.targets[i] for i in remaining],  # Again keep the same mix of flower types.
)

train_data = Subset(train_view, train_indices)   # Training photos (with random flips/rotations).
val_data = Subset(eval_view, val_indices)        # Validation photos (no random changes).
test_data = Subset(eval_view, test_indices)      # Test photos (no random changes).
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)  # Training photos in batches of 20, new random order each epoch.
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, num_workers=0)    # Validation photos in batches of 20.
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=0)  # Test photos in batches of 20 (all 551 of them).

print('Training:', len(train_data), '| Validation:', len(val_data), '| Test:', len(test_data))  # 2569 | 550 | 551
images, labels = next(iter(train_loader))        # Grab one batch to check its shape.
print('One batch:', images.shape, '| Labels:', labels.shape)  # [20 photos, 3 colors, 150, 150] and 20 labels.

## 5. Reuse VGG16 features and add our classifier

VGG16 has two main parts: convolutional layers that extract features, and a classifier that combines them into predictions. We keep the pretrained `.features` and replace the original classifier with a small **flower** classifier.

Our model: VGG16 features → Flatten → 256 neurons → ReLU → Dropout → **5 outputs** (one per flower class).

- `requires_grad = False` freezes the pretrained weights: they are used but not updated.
- A dummy image finds the feature-output size. At 150 × 150 input size, it is 512 × 4 × 4, or 8,192 numbers after flattening.
- The new Linear layers start with randomly initialized weights.
- ReLU adds nonlinearity. Dropout temporarily disables some activations during training to help reduce overfitting.
- **Change for flowers:** the final layer outputs five raw scores (logits), one per class. The predicted class is the one with the highest score.
- `CrossEntropyLoss` replaces `BCEWithLogitsLoss` because there are five classes instead of two. It applies softmax internally, so do not add softmax before this loss; use softmax later when displaying probabilities.

In [ ]:
pretrained = models.vgg16(weights=models.VGG16_Weights.DEFAULT)  # Download VGG16, already trained on ImageNet.
conv_base = pretrained.features                  # Keep only its convolution layers (the part that recognises patterns).
del pretrained  # Release the large original classifier that we do not need.

# Freeze all pretrained convolution weights for stage 1.
for parameter in conv_base.parameters():         # For every weight in VGG16's convolution layers:
    parameter.requires_grad = False              # freeze it (it is used, but training will not change it).

with torch.no_grad():                            # Quick test run, no training:
    feature_shape = conv_base(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)).shape  # pass one blank image to see VGG16's output size.
    # Count the values from one image after flattening.
    number_of_features = feature_shape[1] * feature_shape[2] * feature_shape[3]  # 512 x 4 x 4 = 8,192 numbers per image.

model = nn.Sequential(                           # The full model, applied in this order:
    conv_base,                                   # 1. VGG16 turns the photo into 512 x 4 x 4 pattern scores.
    nn.Flatten(),                                # 2. Line them up into one list of 8,192 numbers.
    nn.Linear(number_of_features, 256),          # 3. New layer: 8,192 numbers -> 256 numbers.
    nn.ReLU(),                                   # 4. Set negative values to 0.
    nn.Dropout(0.5),                             # 5. During training, randomly switch off half of the 256 (reduces overfitting).
    nn.Linear(256, num_classes),  # 6. New layer: 256 -> 5 scores, one per flower class (highest score = prediction).
).to(device)                                     # Put the model on the GPU (or CPU).
loss_function = nn.CrossEntropyLoss()            # Measures how wrong the 5 scores are vs. the true class (used for 3+ classes).
print('Convolution output:', tuple(feature_shape))  # (1, 512, 4, 4)
print('Flattened features:', number_of_features)    # 8192
print('Total parameters:', sum(p.numel() for p in model.parameters()))                          # All weights in the model.
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))   # Weights that will learn (only the 2 new layers).

## 6. Define what happens during one pass through the data

This is the only batch-processing loop in the program. We reuse it for training, validation, and testing.

If an optimizer is supplied, we turn on training behavior, calculate gradients, and update weights. Otherwise we use evaluation mode without gradients. Dropout is active only during training. This VGG16 variant has no BatchNorm layers.

**Changes for flowers:** the model returns five logits per image, so there is no `squeeze(1)`. `CrossEntropyLoss` needs integer (`long`) labels. The predicted class is `logits.argmax(1)`, the class with the highest score.

In [ ]:
def run_epoch(loader, optimizer=None):           # Go through all photos in `loader` once. With an optimizer it trains; without, it only measures.
    training = optimizer is not None             # True = training pass, False = validation/test pass.
    model.train(training)  # Training mode turns Dropout on; evaluation mode turns it off.
    total_loss, correct, count = 0.0, 0, 0       # Running totals, start at zero.
    # Gradients are needed for training but not for validation or testing.
    with torch.set_grad_enabled(training):
        for images, labels in loader:            # Take one batch of 20 photos at a time:
            images = images.to(device)           # put the photos on the GPU,
            labels = labels.to(device)           # and their true classes (numbers 0-4).
            if training:                         # Only when training:
                optimizer.zero_grad()                  # clear the previous batch's weight adjustments.
            logits = model(images)                      # Model gives 5 scores per photo.
            loss = loss_function(logits, labels)        # How wrong were the scores? (one number for the batch)
            if training:                         # Only when training:
                loss.backward()                        # work out how each weight should change,
                optimizer.step()                       # and change the weights a little.
            total_loss += loss.item() * len(labels)     # Add up the loss (times batch size, so the last smaller batch counts correctly).
            correct += (logits.argmax(1) == labels).sum().item()  # Count photos where the highest score is the true class.
            count += len(labels)                        # Count photos seen.
    return total_loss / count, correct / count   # Average loss and accuracy over all photos.

## 7. Repeat training and keep the best weights

`fit()` calls `run_epoch()` once for training and once for validation each epoch. It records losses and accuracies for each epoch.

Each call creates a new optimizer using only trainable parameters. This is important when stage 2 unfreezes additional layers.

We save a copy of the weights whenever validation loss improves. Early stopping ends a stage after three consecutive epochs without improvement. At the end, we restore the best weights, not necessarily the last epoch's weights.

The starting model is also considered. This protects the stage-1 model if fine-tuning does not improve validation loss. "Epoch 0" means the starting weights remained best.

In [ ]:
def fit(epochs, learning_rate, stage_name):      # Train for up to `epochs` epochs and keep the best version of the model.
    # Create a new optimizer after changing which parameters are frozen.
    optimizer = torch.optim.RMSprop(             # The optimizer is what updates the weights.
        [p for p in model.parameters() if p.requires_grad], lr=learning_rate,  # Give it only the unfrozen weights.
        alpha=0.9, eps=1e-7,  # Technical settings of RMSprop (kept from the class notebook).
    )
    history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}  # Lists to record each epoch's results.

    # Evaluate the incoming model before changing any weights.
    best_loss, _ = run_epoch(val_loader)         # Validation loss BEFORE this stage's training (the score to beat).
    best_weights = copy.deepcopy(model.state_dict())  # Save the starting weights as the best so far.
    best_epoch, waiting = 0, 0                   # Best epoch so far (0 = starting weights); epochs since last improvement.

    for epoch in range(1, epochs + 1):           # For each epoch:
        train_loss, train_accuracy = run_epoch(train_loader, optimizer)  # 1. Train on all training photos.
        val_loss, val_accuracy = run_epoch(val_loader)                   # 2. Measure on the validation photos (no learning).
        history['train_loss'].append(train_loss)          # 3. Record this epoch's training loss,
        history['val_loss'].append(val_loss)              #    validation loss,
        history['train_accuracy'].append(train_accuracy)  #    training accuracy,
        history['val_accuracy'].append(val_accuracy)      #    and validation accuracy.

        # Select weights using validation loss, never test results.
        if val_loss < best_loss:                 # 4. If validation loss is the best so far:
            best_loss = val_loss                 #    remember it,
            best_weights = copy.deepcopy(model.state_dict())  #    save a copy of these weights,
            best_epoch, waiting = epoch, 0       #    and reset the waiting counter.
        else:                                    #    Otherwise:
            waiting += 1                         #    one more epoch without improvement.

        print(f'{stage_name} {epoch}/{epochs} | Train: {train_accuracy:.2%}' # 5. Show progress: training accuracy,
              f' | Validation: {val_accuracy:.2%} | Val loss: {val_loss:.4f}', flush=True) #    validation accuracy and validation loss.
        if waiting >= PATIENCE:                  # 6. No improvement for 3 epochs in a row:
            print('Early stopping: validation loss stopped improving.') #    say so,
            break                                #    stop this stage early.

    # Undo any later epochs that performed worse on validation data.
    model.load_state_dict(best_weights)          # Go back to the best weights found.
    print(f'{stage_name}: restored best weights from epoch {best_epoch}.') # Say which epoch's weights are kept.
    return history                               # Give back the recorded results.

## 8. Stage 1 — train only the new classifier

The convolutional base is frozen. Only the new Linear layers learn, while Dropout discourages reliance on individual hidden features. The validation result printed after fit belongs to the restored best model, not necessarily the final epoch shown in the history.

**Added for Problem 3:** we keep a copy of the stage-1 weights (`stage1_weights`) so that the stage-1 model can be evaluated on the test set in section 10, after stage 2 has changed the model.

In [ ]:
history_initial = fit(INITIAL_EPOCHS, INITIAL_LR, 'Frozen base')  # Stage 1: VGG16 frozen, only the 2 new layers learn.
initial_val_loss, initial_val_accuracy = run_epoch(val_loader)    # Validation score of the best stage-1 model.
print(f'Selected stage-1 validation accuracy: {initial_val_accuracy:.2%}') # Show it.
stage1_weights = copy.deepcopy(model.state_dict())  # Save the stage-1 model now; we test it in section 10 (stage 2 will change it).

## 9. Stage 2 — fine-tune the final convolutional block

VGG16 has five convolutional blocks. `conv_base[24:]` selects block 5, starting at module index 24. It contains three convolutions, their ReLUs, and pooling.

We make this block trainable while keeping blocks 1–4 frozen. The classifier also remains trainable. We reuse the weights learned in stage 1; we do not create a new model.

Calling `fit()` again creates an optimizer that includes the newly unfrozen parameters. The smaller learning rate helps adjust pretrained features without changing them too abruptly.

In [ ]:
# Module index 24 starts block 5 (it is not the 24th convolution).
for parameter in conv_base[24:].parameters():    # For every weight in VGG16's last block (block 5):
    parameter.requires_grad = True               # unfreeze it so it can learn. Blocks 1-4 stay frozen.

print('Trainable convolution modules:')           # Heading for the list below.
for index, layer in enumerate(conv_base):        # Go through VGG16's layers
    if isinstance(layer, nn.Conv2d) and layer.weight.requires_grad:  # and list the convolution layers that can now learn
        print(f'  features[{index}]: {layer}')   # (should be the 3 in block 5).
print('Total trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))  # Now block 5 + the 2 new layers.

history_finetune = fit(FINETUNE_EPOCHS, FINETUNE_LR, 'Fine-tuning')  # Stage 2: continue training, starting from the stage-1 model.
final_val_loss, final_val_accuracy = run_epoch(val_loader)           # Validation score of the best stage-2 model.
print(f'Selected final validation accuracy: {final_val_accuracy:.2%}') # Show it.
stage2_weights = copy.deepcopy(model.state_dict())  # Save the stage-2 model (we switch between the two in section 10).

## 10. Evaluate the stage 1 and stage 2 models on the reserved test set

The test images are now used for the first time.

**Problem 3:** we evaluate **both** models on the **entire test set** (all 551 images): first the stage-1 weights saved in section 8, then the final stage-2 weights. For each, `test_report()` prints the classification report and computes **overall precision**:

- **Macro precision:** the average of the five per-class precisions (each class counts equally).
- **Weighted precision:** each class weighted by its number of test images.

Precision asks how often a predicted class is correct. Recall asks how many actual members of a class were found. F1 combines precision and recall; support is the number of images. Do not tune settings using this test report.

In [ ]:
def test_report(stage_name):                     # Test the current model on ALL 551 test photos and print the results.
    test_loss, test_accuracy = run_epoch(test_loader)  # Test loss and accuracy (no learning).
    actual_labels, predictions = [], []          # Lists for the true class and the predicted class of each test photo.
    model.eval()                                 # Evaluation mode (Dropout off).
    with torch.no_grad():                        # No training here.
        for images, labels in test_loader:       # For each batch of test photos:
            logits = model(images.to(device))    # get the 5 scores per photo,
            predictions.extend(logits.argmax(1).cpu().tolist())  # predicted class = the one with the highest score,
            actual_labels.extend(labels.tolist())                # and note the true class.
    assert len(actual_labels) == len(test_data)  # Check that all 551 test photos were used.
    precision_macro = precision_score(actual_labels, predictions, average='macro', zero_division=0)        # Overall precision: average of the 5 class precisions.
    precision_weighted = precision_score(actual_labels, predictions, average='weighted', zero_division=0)  # Same, but bigger classes count more.
    print(f'===== {stage_name} | test images evaluated: {len(actual_labels)} of {len(test_data)} =====')  # Header.
    print(f'Test accuracy: {test_accuracy:.2%} | Test loss: {test_loss:.4f}')                                # Accuracy and loss.
    print(f'Overall precision (macro): {precision_macro:.2%} | Overall precision (weighted): {precision_weighted:.2%}')  # Overall precision.
    print(classification_report(actual_labels, predictions, labels=list(range(num_classes)), # Print the per-class table
                                target_names=class_names, zero_division=0))  # Table: precision, recall, F1 for each flower class.
    return {'test_accuracy': test_accuracy, 'test_loss': test_loss, # Give back accuracy, loss,
            'precision_macro': precision_macro, 'precision_weighted': precision_weighted}  # Return the numbers for the comparison table.

model.load_state_dict(stage1_weights)            # Load the saved stage-1 model...
stage1_results = test_report('Stage 1 (frozen base)')  # ...and test it.
model.load_state_dict(stage2_weights)            # Load the saved stage-2 (final) model...
stage2_results = test_report('Stage 2 (fine-tuned)')   # ...and test it.

print(f"{'Model':<24}{'Precision (macro)':>19}{'Precision (weighted)':>22}{'Accuracy':>10}")  # Comparison table: header row.
for stage_name, results in [('Stage 1 (frozen base)', stage1_results), ('Stage 2 (fine-tuned)', stage2_results)]:  # One row per stage:
    print(f"{stage_name:<24}{results['precision_macro']:>19.2%}{results['precision_weighted']:>22.2%}" # Print one row:
          f"{results['test_accuracy']:>10.2%}")  # name, macro precision, weighted precision, accuracy.

## 11. Predict one image of every flower class

**Problem 3:** we test the final (stage 2) model with **one test image of each of the five flower classes**: the first test image of each class.

Apply the same validation/test preprocessing to each photo. `unsqueeze(0)` adds the batch dimension because the model expects a batch, even for one image.

Softmax converts the five output logits to class probabilities that add up to 1. These are model estimates, not guarantees. The title is green when the prediction is correct and red when it is wrong.

In [ ]:
# Use the first test image of each class to demonstrate prediction on single photos.
fig, axes = plt.subplots(1, num_classes, figsize=(4 * num_classes, 4))  # A row of 5 empty panels, one per flower class.
model.eval()                                     # Evaluation mode (Dropout off). The model here is the final stage-2 model.
for axis, class_index in zip(axes, range(num_classes)):  # For each flower class (0 = daisy ... 4 = tulips):
    photo_index = next(i for i in test_indices if eval_view.targets[i] == class_index)  # find the first test photo of that class,
    photo_path, actual_label = eval_view.samples[photo_index]  # get its file path and true class.
    with Image.open(photo_path) as photo:        # Open the photo file.
        photo = photo.convert('RGB')             # Make sure it has 3 color channels.
        input_image = eval_transform(photo).unsqueeze(0).to(device)  # Same preprocessing as test photos; unsqueeze(0) makes it a batch of 1.
        with torch.no_grad():                    # No training here.
            probabilities = torch.softmax(model(input_image), dim=1)[0]  # Turn the 5 scores into 5 probabilities that add up to 100%.
        predicted_label = probabilities.argmax().item()  # Predicted class = the one with the highest probability.
        axis.imshow(photo)                       # Show the photo in its panel.
    axis.set_title(f'Actual: {class_names[actual_label]}\nPredicted: {class_names[predicted_label]}' # Panel title: true class,
                   f' ({probabilities[predicted_label]:.1%})', # predicted class and its probability,
                   color='green' if predicted_label == actual_label else 'red')  # Title: true vs predicted class; green = right, red = wrong.
    axis.axis('off')                             # Hide the axis numbers.
    print(f'{class_names[actual_label]:<11} -> predicted {class_names[predicted_label]:<11} | probabilities: ' # Print: true class -> predicted class,
          + ', '.join(f'{name} {p:.1%}' for name, p in zip(class_names, probabilities.tolist())))  # Also print all 5 probabilities.
fig.suptitle('Final model: one test image of every flower class')  # Title for the whole figure.
plt.tight_layout()                               # Reduce overlap.
plt.show()                                       # Display the figure.

## 12. Report on results

_Fill in this section after running the notebook, using the outputs of sections 10 and 11._

**Precision on the full test set (551 images)**

| Model | Precision (macro) | Precision (weighted) | Test accuracy |
|:---|:---:|:---:|:---:|
| Stage 1 (frozen VGG16 base) | | | |
| Stage 2 (fine-tuned block 5) | | | |

Points to discuss:

1. Did fine-tuning (stage 2) improve precision over stage 1, and by how much?
2. Which flower classes have the highest and lowest precision?
3. How did the five single-image tests in section 11 go?
4. How does VGG16 transfer learning compare with the small CNN from Problem 2 (best: 75.8% macro precision)?